In [30]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1oGLLTtudUdfeN0iF8HDgRsqPvjoaw42CvvIa8EDgQ1w"
SHEET_NAME = "update june"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()


# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [31]:
duck.sql(
    f"""
    create or replace table ez as 
SELECT * FROM read_gsheet(
    '{_sql_literal(SPREADSHEET_ID)}',
    sheet='{_sql_literal(SHEET_NAME)}',
    all_varchar=true
)
"""
)

In [25]:
duck.sql(
    """
    select "Kontakt: Kundennummer", coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name")) as easybill_firm_name, a.Id as zoho_id, a."Account_Name" as zoho_name
    from pg.easybill.contacts
    left join pg.zoho.Accounts a 
        on a.Kundenummer = "Kontakt: Kundennummer"
    where "Kontakt: Kundennummer"::varchar
     not in (
        select easybill_kundennummer::varchar from ez)

    """
).to_csv('update_easybill_zoho_rostock.csv')

In [29]:
duck.sql(
    """
    select * from pg.bas_firms.easybill_zoho
    
    """
)

┌───────┬─────────────┬────────────────────┐
│  id   │ easybill_id │      zoho_id       │
│ int32 │   varchar   │      varchar       │
├───────┼─────────────┼────────────────────┤
│ 45747 │ 130002466   │ 386758000009962010 │
│ 45748 │ 105000002   │ 386758000009963148 │
│ 45749 │ 130002395   │ 386758000009995156 │
│ 45750 │ 10602 0027  │ 386758000010018411 │
│ 45751 │ 106020037   │ 386758000010037523 │
│ 45752 │ 107030007   │ 386758000010037554 │
│ 45753 │ 105010052   │ 386758000010038012 │
│ 45754 │ 101000025   │ 386758000010040289 │
│ 45755 │ 100020030   │ 386758000010040350 │
│ 45756 │ 106020036   │ 386758000010043966 │
│   ·   │     ·       │  ·                 │
│   ·   │     ·       │  ·                 │
│   ·   │     ·       │  ·                 │
│ 12976 │ 130001038   │ NULL               │
│ 12977 │ 130001933   │ NULL               │
│ 12978 │ 119020052   │ NULL               │
│ 12979 │ 130000941   │ NULL               │
│ 12980 │ 116010039   │ NULL               │
│ 12981 │ 

In [35]:
duck.sql(
    """
    select * from ez where "Kontakt: Kundennummer"  in (select easybill_id from pg.bas_firms.easybill_zoho)
    """
)

┌───────────────────────┬──────────────────────────────────────────────────────────────────────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────┐
│ Kontakt: Kundennummer │                            easybill_firm_name                            │      zoho_id       │                                zoho_name                                 │
│        varchar        │                                 varchar                                  │      varchar       │                                 varchar                                  │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────┼────────────────────┼──────────────────────────────────────────────────────────────────────────┤
│ 130002406             │ Medicover Ulm MVZ GmbH                                                   │ 386758000068083626 │ Medicover Ulm                                                            │
│ 130002467    

In [ ]:
# Insert NEW rows from the "update june" sheet into bas_firms.easybill_zoho.
# - Anti-join on easybill_id => only rows whose Kundennummer is not yet in the
#   table are inserted (existing easybill_ids are left untouched). Idempotent:
#   re-running inserts nothing.
# - `id` is left to the bas_firms.easybill_zoho_id_seq sequence.
# - zoho_id "no id" (no Zoho match) is stored as NULL, matching the existing
#   convention in the table.
before = duck.sql("select count(*) from pg.bas_firms.easybill_zoho").fetchone()[0]

duck.execute(
    """
    INSERT INTO pg.bas_firms.easybill_zoho (easybill_id, zoho_id)
    SELECT ez."Kontakt: Kundennummer",
           CASE WHEN ez.zoho_id ~ '^[0-9]+$' THEN ez.zoho_id ELSE NULL END
    FROM ez
    LEFT JOIN pg.bas_firms.easybill_zoho t
           ON t.easybill_id = ez."Kontakt: Kundennummer"
    WHERE t.easybill_id IS NULL
    """
)

after = duck.sql("select count(*) from pg.bas_firms.easybill_zoho").fetchone()[0]
print(f"inserted {after - before} new rows ({before} -> {after})")